In [1]:
%pip install --no-build-isolation --quiet anthropic==0.109.1 python-dotenv anthropic[bedrock] boto3==1.37.3 

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: C:\Program Files\Python313\python.exe -m pip install --upgrade pip


In [2]:
from dotenv import load_dotenv

load_dotenv()

True

In [3]:
from anthropic import AnthropicBedrock
import os
import boto3

client = AnthropicBedrock(
    aws_access_key=os.environ["AWS_ACCESS_KEY_ID"],
    aws_secret_key=os.environ["AWS_SECRET_ACCESS_KEY"],
    aws_session_token=os.environ.get("AWS_SESSION_TOKEN"),  # optional
    aws_region=os.environ["AWS_REGION"],
)
model = "us.anthropic.claude-sonnet-4-20250514-v1:0"


In [6]:
import json

def add_user_message(messages, text):
    user_message = {
            "role": "user",
            "content": text
        }
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {
            "role": "assistant",
            "content": text
        }
    messages.append(assistant_message)

def chat(messages, system=None, temperature=0.0, stop_sequences=[]):
    params = {

        "model" : model,
        "max_tokens" : 1000,
        "messages" : messages,
        "temperature": temperature
    }

    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    message = client.messages.create(**params)

    return message.content[0].text

def generate_dataset():

    prompt = """
    Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, 
    JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, 
    JSON, or a Regex to complete.

    Example output:
    ```json
    [
      {
        "task": "Description of task",
      },
      ...additional
    ]
    ```

    * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
    * Focus on tasks that do not require writing much code
    
    Please generate 3 objects.

    """

    messages = []
    add_user_message(messages, prompt)

    add_assistant_message(messages, "```json")
    text = chat(messages, stop_sequences = ["```"])
    return json.loads(text)

from IPython.display import display, Markdown
def print_response(message):
    print("-" * 50)
    display(Markdown(message))
    print("-" * 50)
    

In [9]:
dataset = generate_dataset()

with open('dataset.json' , 'w') as f:
    json.dump(dataset, f, indent = 2)

In [31]:
def run_prompt(test_case):
    """Merges the prompt and test case input and then returns the result"""
    prompt = f"""
    Please solve the following task:
    
    {test_case["task"]}
    """

    messages = []
    add_user_message(messages, prompt)
    output = chat(messages)
    return output

def run_test_case(test_case):
    """Call run_prompt then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading

    model_grade = grade_by_model(test_case, output)
    score = model_grade["score"]
    reasoning = model_grade["reasoning"]

    return{
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

def grade_by_model(test_case, output):
    eval_prompt = f"""
    You are an export code reviewer. Evaluate this AI generated solution.

    Original Task: 
    <task>
    {test_case["task"]}
    </task>

    Solution To Evaluate
    <solution>
    {output}
    </solution>

    Output Format
    provide your evaluation as a JSON object with:
    - "strenghts" : An array of 1 - 3 key strenghts
    - "weaknesses": An array of 1 - 3 key areas of improvement
    - "reasoning" : A consise explanation of your assessment
    - "score"     : A number between 1 and 10

    {{
        "strenghts": string[],
        "weaknesses": string[],
        "reasoning": string,
        "score": number,
        
    }}
    

    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results


In [34]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

In [35]:
print(json.dumps(results, indent = 2))

[
  {
    "output": "Here's a Python function that extracts the bucket name from an AWS S3 bucket ARN:\n\n```python\ndef extract_bucket_name_from_arn(arn):\n    \"\"\"\n    Extract the bucket name from an AWS S3 bucket ARN.\n    \n    Args:\n        arn (str): The S3 bucket ARN in format 'arn:aws:s3:::bucket-name'\n    \n    Returns:\n        str: The bucket name\n    \n    Raises:\n        ValueError: If the ARN format is invalid\n    \"\"\"\n    if not isinstance(arn, str):\n        raise ValueError(\"ARN must be a string\")\n    \n    # Split the ARN by colons\n    parts = arn.split(':')\n    \n    # Validate ARN format\n    if len(parts) != 6:\n        raise ValueError(\"Invalid ARN format. Expected 6 parts separated by colons\")\n    \n    if parts[0] != 'arn':\n        raise ValueError(\"ARN must start with 'arn'\")\n    \n    if parts[1] != 'aws':\n        raise ValueError(\"Invalid partition. Expected 'aws'\")\n    \n    if parts[2] != 's3':\n        raise ValueError(\"Invalid 